# ParaKoop — DrivAerNet OpenFOAM Validation

Runs 3–5 real DrivAerNet car STL files through OpenFOAM simpleFoam (k-ω SST RANS)
and compares CFD Cd against:
1. DrivAerNet ground-truth Cd (from `geometry_features.csv`)
2. ParaKoop model predictions

**Runtime**: ~1–2 hrs per case on Colab CPU (GPU not used by OpenFOAM)

**Prerequisites**:
- Mount Google Drive containing `parakoop/` repo and DrivAerNet STL zips
- Colab compute credits (High-RAM recommended)

## 1. Mount Drive & Clone/Pull Repo

In [86]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Clone latest code from GitHub
if not os.path.exists('/content/parakoop'):
    !git clone https://github.com/Ranaam21/Parakoop.git /content/parakoop
else:
    !git -C /content/parakoop pull origin main

PARAKOOP_DIR = '/content/parakoop'

# Data and checkpoint stay in Drive (not tracked by git)
DATA_DIR = '/content/drive/MyDrive/Car_CFD/parakoop/data/drivaernet'
CKPT     = '/content/drive/MyDrive/Car_CFD/parakoop/checkpoints/unified/parakoop_unified_best.pt'

os.chdir(PARAKOOP_DIR)
print('Repo ready :', PARAKOOP_DIR)
print('Data dir   :', DATA_DIR)
print('Checkpoint :', CKPT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.20 KiB | 409.00 KiB/s, done.
From https://github.com/Ranaam21/Parakoop
 * branch            main       -> FETCH_HEAD
   02d8620..ef7beb8  main       -> origin/main
Updating 02d8620..ef7beb8
Fast-forward
 notebooks/drivaernet_openfoam_validation.ipynb | 8 ++++++++
 openfoam/drivaernet_case_generator.py          | 3 ++-
 2 files changed, 10 insertions(+), 1 deletion(-)
Repo ready : /content/parakoop
Data dir   : /content/drive/MyDrive/Car_CFD/parakoop/data/drivaernet
Checkpoint : /content/drive/MyDrive/Car_CFD/parakoop/checkpoints/unified/parakoop_unified_best.pt


## 2. Install OpenFOAM v2312

In [87]:
%%bash
# Add OpenFOAM repository and install
curl -s https://dl.openfoam.com/add-debian-repo.sh | sudo bash
sudo apt-get update -qq
sudo apt-get install -y openfoam2312 2>&1 | tail -5
echo 'OpenFOAM installed'

Detected distribution code-name: jammy
Overwrote /etc/apt/sources.list.d/openfoam.list
Importing openfoam gpg key... done
Overwrote /etc/apt/trusted.gpg.d/openfoam.gpg
Running apt-get update... done

The repository is setup! You can now install packages.
Reading package lists...
Building dependency tree...
Reading state information...
openfoam2312 is already the newest version (2312.260127-2).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.
OpenFOAM installed


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [88]:
# Source OpenFOAM environment for all subsequent bash cells
import subprocess
OF_BASHRC = '/usr/lib/openfoam/openfoam2312/etc/bashrc'
result = subprocess.run(['bash', '-c', f'source {OF_BASHRC} && env'],
                        capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if line.startswith(('FOAM_', 'WM_', 'PATH=', 'LD_LIBRARY')):
        key, _, val = line.partition('=')
        os.environ[key] = val
print('OpenFOAM env loaded. Version:', os.environ.get('WM_PROJECT_VERSION', 'unknown'))

OpenFOAM env loaded. Version: v2312


## 3. Install Python Dependencies & Load Model

In [89]:
!pip install -q torch numpy pandas scikit-learn scipy tqdm

import sys
sys.path.insert(0, PARAKOOP_DIR)

import torch
import pandas as pd
import numpy as np
from koopman.model import ParaKoopModel
from data_pipeline.unified_loader import load_unified, THETA_COLS

# CKPT defined in Cell 1 — reads from Drive
ckpt  = torch.load(CKPT, map_location='cpu', weights_only=False)
cfg   = ckpt.get('model_cfg', {})
model = ParaKoopModel(
    phi_dim=cfg.get('phi_dim', 1),
    theta_dim=cfg.get('theta_dim', 8),
    koopman_dim=cfg.get('koopman_dim', 128),
    operator_rank=cfg.get('operator_rank', 16),
    hidden_lift=cfg.get('hidden_lift', 64),
    hidden_op=cfg.get('hidden_op', 64),
    lambda_fp_cd=cfg.get('lambda_fp_cd', 0.1),
)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Model loaded  : {sum(p.numel() for p in model.parameters()):,} params')
print(f'Checkpoint    : {CKPT}')

Model loaded  : 14,884,434 params
Checkpoint    : /content/drive/MyDrive/Car_CFD/parakoop/checkpoints/unified/parakoop_unified_best.pt


## 4. Select DrivAerNet STL Cases

In [90]:
import zipfile, shutil, glob

# Load geometry features
geo_df   = pd.read_csv(os.path.join(DATA_DIR, 'geometry_features.csv'))
MESH_DIR = os.path.join(DATA_DIR, 'meshes')

# Auto-discover every zip in the meshes folder
all_zips = sorted(glob.glob(os.path.join(MESH_DIR, '*.zip')))
print(f'Found {len(all_zips)} zip(s) in {MESH_DIR}')

CASES = []
for zip_path in all_zips:
    zip_name   = os.path.basename(zip_path)
    config_key = zip_name.replace('.zip', '')          # e.g. E_S_WW_WM

    # Classify body style from first letter of config
    first = config_key[0]
    style = {'E': 'estateback', 'F': 'fastback', 'N': 'notchback'}.get(first, 'unknown')

    # Validate zip
    try:
        with zipfile.ZipFile(zip_path) as z:
            z.namelist()
    except Exception as e:
        print(f'  SKIP {zip_name}: corrupt ({e})  ← re-upload')
        continue

    # Find rows in geometry_features.csv matching this config
    subset = geo_df[geo_df['design_id'].str.startswith(config_key)].copy()
    if len(subset) == 0:
        print(f'  SKIP {zip_name}: no matching rows in geometry_features.csv')
        continue

    # Pick median-Cd design for this config
    median_cd = subset['cd'].median()
    row = subset.iloc[(subset['cd'] - median_cd).abs().argsort()[:1]].iloc[0]

    CASES.append({
        'zip':       zip_name,
        'style':     style,
        'design_id': row['design_id'],
        'cd_csv':    float(row['cd']),
        'geo':       row.to_dict(),
    })
    print(f'  {style:12s}  {config_key:<20s}  design={row["design_id"]}  Cd_csv={row["cd"]:.4f}')

print(f'\n{len(CASES)} case(s) queued  '
      f'(est. runtime: {len(CASES)*90//60}–{len(CASES)*2} hrs on Colab CPU)')

Found 9 zip(s) in /content/drive/MyDrive/Car_CFD/parakoop/data/drivaernet/meshes
  SKIP E_S_WWC_WM.zip: corrupt (File is not a zip file)  ← re-upload
  estateback    E_S_WW_WM             design=E_S_WW_WM_0555  Cd_csv=0.2739
  SKIP F_D_WM_WW_1.zip: corrupt (File is not a zip file)  ← re-upload
  SKIP F_D_WM_WW_2.zip: corrupt (File is not a zip file)  ← re-upload
  fastback      F_S_WWC_WM            design=F_S_WWC_WM_0528  Cd_csv=0.2459
  fastback      F_S_WWS_WM            design=F_S_WWS_WM_0590  Cd_csv=0.2420
  SKIP N_S_WWC_WM.zip: corrupt (File is not a zip file)  ← re-upload
  SKIP N_S_WWS_WM.zip: corrupt (File is not a zip file)  ← re-upload
  SKIP N_S_WW_WM.zip: corrupt (File is not a zip file)  ← re-upload

3 case(s) queued  (est. runtime: 4–6 hrs on Colab CPU)


In [91]:
STL_DIR = '/content/drivaernet_stls'
os.makedirs(STL_DIR, exist_ok=True)

def _zip_stl_name(design_id: str) -> str:
    """
    Convert CSV design_id (4-digit suffix) to zip filename (3-digit suffix).
    E_S_WW_WM_0555  →  E_S_WW_WM_555.stl
    E_S_WW_WM_0012  →  E_S_WW_WM_012.stl
    """
    parts  = design_id.rsplit('_', 1)
    config = parts[0]                       # E_S_WW_WM
    idx    = int(parts[1])                  # 555
    return f"{config}_{idx:03d}.stl"

for c in CASES:
    zip_path = os.path.join(MESH_DIR, c['zip'])
    zip_stl  = _zip_stl_name(c['design_id'])   # name inside zip
    out_name = c['design_id'] + '.stl'          # local name we save as
    out_path = os.path.join(STL_DIR, out_name)

    if not os.path.exists(out_path):
        print(f"Extracting {zip_stl} from {c['zip']}...")
        with zipfile.ZipFile(zip_path) as z:
            all_names = z.namelist()
            # exact match first, then partial
            matches = [n for n in all_names if n == zip_stl or n.endswith('/' + zip_stl)]
            if not matches:
                # fallback: any file whose basename matches (handles subdirectory layouts)
                matches = [n for n in all_names if os.path.basename(n) == zip_stl]
            if matches:
                extracted = z.extract(matches[0], STL_DIR)
                if extracted != out_path:
                    shutil.move(extracted, out_path)
                print(f'  → {out_name}  ({os.path.getsize(out_path)/1e6:.0f} MB)')
            else:
                print(f'  ERROR: {zip_stl} not in zip. Available sample:')
                for n in all_names[:5]:
                    print(f'    {n}')
                c['stl_path'] = None
                continue
    else:
        print(f'{out_name} already extracted')
    c['stl_path'] = out_path

CASES = [c for c in CASES if c.get('stl_path')]
print(f'\n{len(CASES)} STL(s) ready')

E_S_WW_WM_0555.stl already extracted
F_S_WWC_WM_0528.stl already extracted
F_S_WWS_WM_0590.stl already extracted

3 STL(s) ready


## 5. Generate & Run OpenFOAM Cases

In [ ]:
import sys, importlib
sys.path.insert(0, PARAKOOP_DIR)
import openfoam.drivaernet_case_generator as _dcg
importlib.reload(_dcg)
from openfoam.drivaernet_case_generator import generate_drivaernet_case

# Store cases in Drive — survives any Colab disconnect
CASES_DIR = '/content/drive/MyDrive/Car_CFD/parakoop/of_cases'
os.makedirs(CASES_DIR, exist_ok=True)

NU = 1.516e-5

for c in CASES:
    case_dir = os.path.join(CASES_DIR, c['design_id'])
    generate_drivaernet_case(
        stl_path   = c['stl_path'],
        geo        = c['geo'],
        output_dir = case_dir,
    )
    # Ensure transportModel is present (required by OF 2312)
    tp = os.path.join(case_dir, 'constant', 'transportProperties')
    with open(tp, 'w') as f:
        f.write(f"""FoamFile
{{
    version     2.0;
    format      ascii;
    class       dictionary;
    object      transportProperties;
}}
// * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * //

transportModel  Newtonian;
nu              {NU};
""")
    c['case_dir'] = case_dir
    print(f"Ready: {c['design_id']} → {case_dir}")

In [ ]:
import subprocess, time, os, glob

def has_results(case_dir):
    """True if simpleFoam already completed and wrote force coefficients."""
    coeffs = glob.glob(os.path.join(case_dir, 'postProcessing', '**', '*.dat'), recursive=True)
    return len(coeffs) > 0

def mesh_exists(case_dir):
    """True if snappyHexMesh already ran (mesh is in Drive)."""
    return os.path.exists(os.path.join(case_dir, 'constant', 'polyMesh', 'faces'))

def run_openfoam_case(case_dir):
    """Serial simpleFoam — skips mesh steps if already done, skips entirely if results exist."""
    OF_ENV = 'source /usr/lib/openfoam/openfoam2312/etc/bashrc && '

    if has_results(case_dir):
        print('  SKIP — results already in Drive ✓')
        return True

    if mesh_exists(case_dir):
        # Mesh already in Drive — only run solver
        steps = [('simpleFoam', 'simpleFoam')]
        print('  Mesh found in Drive — running solver only')
    else:
        steps = [
            ('clean',                'find constant/polyMesh -maxdepth 1 -type f ! -name blockMeshDict -delete 2>/dev/null; true'),
            ('blockMesh',            'blockMesh'),
            ('surfaceFeatureExtract','surfaceFeatureExtract'),
            ('snappyHexMesh',        'snappyHexMesh -overwrite'),
            ('simpleFoam',           'simpleFoam'),
        ]

    for name, cmd in steps:
        print(f'  [{name}]', end=' ', flush=True)
        t0 = time.time()
        result = subprocess.run(
            f'cd {case_dir} && {OF_ENV} {cmd} > log.{name} 2>&1',
            shell=True, executable='/bin/bash'
        )
        elapsed = time.time() - t0
        if result.returncode != 0:
            print(f'FAILED ({elapsed:.0f}s)')
            subprocess.run(f'tail -15 {case_dir}/log.{name}', shell=True)
            return False
        print(f'done ({elapsed:.0f}s)')
    return True

for c in CASES:
    print(f"\n{'─'*60}")
    print(f"Case: {c['design_id']}  ({c['style']})")
    c['foam_ok'] = run_openfoam_case(c['case_dir'])

## 6. Extract CFD Results & Compare

In [ ]:
import glob
import numpy as np

THETA_MIN = [0, 0, 0, 0.20, 0.50, 0.00, 0.30, 0.0]
THETA_MAX = [1, 1, 1, 0.45, 1.80, 0.55, 0.70, 1.0]

def geo_to_theta(geo):
    did = str(geo.get('design_id', ''))
    L, H, W = float(geo.get('length_mm', 4700)), float(geo.get('height_mm', 1400)), float(geo.get('width_mm', 2000))
    theta = [
        1.0 if did.startswith('F_') else 0.0,
        1.0 if did.startswith('N_') else 0.0,
        1.0 if did.startswith('E_') else 0.0,
        H / max(L, 1),
        W / max(H, 1),
        float(geo.get('rear_slant_deg', 25)) / 90.0,
        float(geo.get('cabin_length_frac', 0.5)),
        1.0 if '_D_' in did else 0.0,
    ]
    t = torch.tensor(theta, dtype=torch.float32)
    return t.clamp(torch.tensor(THETA_MIN), torch.tensor(THETA_MAX)).unsqueeze(0)

def parse_cd_cl(case_dir, n_avg=100):
    patterns = [
        os.path.join(case_dir, 'postProcessing', '**', 'coefficient.dat'),
        os.path.join(case_dir, 'postProcessing', '**', 'forceCoeffs.dat'),
    ]
    for pat in patterns:
        files = glob.glob(pat, recursive=True)
        if files:
            rows, cd_col, cl_col = [], None, None
            with open(files[0]) as f:
                for line in f:
                    line = line.strip()
                    if line.startswith('#'):
                        parts = line.lstrip('#').split()
                        if 'Cd' in parts: cd_col = parts.index('Cd')
                        if 'Cl' in parts and cl_col is None: cl_col = parts.index('Cl')
                        continue
                    vals = line.split()
                    if len(vals) >= 3:
                        try: rows.append([float(v) for v in vals])
                        except: pass
            if rows:
                arr = np.array(rows)
                tail = arr[-n_avg:]
                cd_col = cd_col if cd_col is not None else 1
                cl_col = cl_col if cl_col is not None else 4
                return float(np.mean(tail[:, cd_col])), float(np.mean(tail[:, cl_col]))
    return None, None

def model_predict(geo):
    t = geo_to_theta(geo)
    with torch.no_grad():
        cd_p, cl_p, _ = model.forward_cd_only(t)
    return float(cd_p), float(cl_p)

records = []
for c in CASES:
    cd_cfd, cl_cfd = (parse_cd_cl(c['case_dir']) if c.get('foam_ok') else (None, None))
    cd_pk,  cl_pk  = model_predict(c['geo'])
    records.append({
        'design_id': c['design_id'], 'style': c['style'],
        'cd_csv': c['cd_csv'],
        'cd_cfd': round(cd_cfd, 4) if cd_cfd is not None else None,
        'cd_pk':  round(cd_pk,  4),
        'cl_cfd': round(cl_cfd, 4) if cl_cfd is not None else None,
        'cl_pk':  round(cl_pk,  4),
    })

df = pd.DataFrame(records)
df['err_csv_vs_pk']  = (df['cd_csv'] - df['cd_pk']).abs()
df['err_cfd_vs_pk']  = (df['cd_cfd'] - df['cd_pk']).abs()
df['err_cfd_vs_csv'] = (df['cd_cfd'] - df['cd_csv']).abs()
print(df[['design_id','style','cd_csv','cd_cfd','cd_pk',
          'err_csv_vs_pk','err_cfd_vs_pk','err_cfd_vs_csv']].to_string(index=False))

# Save to Drive — persists across sessions
RESULTS_DRIVE = '/content/drive/MyDrive/Car_CFD/parakoop/results'
os.makedirs(RESULTS_DRIVE, exist_ok=True)
out_csv = os.path.join(RESULTS_DRIVE, 'drivaernet_cfd_validation.csv')
df.to_csv(out_csv, index=False)
print(f'\nSaved → {out_csv}')

## 7. Summary Table

In [ ]:
print('\n' + '═'*80)
print('  ParaKoop — DrivAerNet CFD Validation  (simpleFoam k-ω SST, U∞=40m/s)')
print('═'*80)
print(f"{'Design':<25} {'Style':<12} {'Cd_csv':>8} {'Cd_cfd':>8} {'Cd_PK':>8} "
      f"{'|Δ|csv':>8} {'|Δ|cfd':>8}")
print('  ' + '─'*74)
for _, r in df.iterrows():
    cfd_s   = f"{r.cd_cfd:.4f}" if r.cd_cfd else '  n/a  '
    ecfd_s  = f"{r.err_cfd_vs_pk:.4f}" if r.cd_cfd else '  n/a  '
    print(f"  {r.design_id:<23} {r.style:<12} {r.cd_csv:>8.4f} "
          f"{cfd_s:>8} {r.cd_pk:>8.4f} {r.err_csv_vs_pk:>8.4f} {ecfd_s:>8}")
print('  ' + '─'*74)
valid = df.dropna(subset=['cd_cfd'])
if len(valid):
    print(f"  Mean |Δ| PK vs CSV : {df.err_csv_vs_pk.mean():.4f}")
    print(f"  Mean |Δ| PK vs CFD : {valid.err_cfd_vs_pk.mean():.4f}")
    print(f"  Mean |Δ| CFD vs CSV: {valid.err_cfd_vs_csv.mean():.4f}  (CFD setup accuracy check)")
print('═'*80)